# Q1c — TF-IDF → two-layer MLP

**W&B run:** _filled in after first run_

Same MLP as Q1b, but the input is now a TF-IDF vector instead of a mean-pooled word2vec vector.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import scipy.sparse as sp
import torch
import wandb
from torch.utils.data import DataLoader, TensorDataset

from nlp_project import SEED, set_seed
from nlp_project.data import load_20ng, preprocess, train_val_split
from nlp_project.eval import evaluate, plot_confusion
from nlp_project.model import MLP
from nlp_project.train import train as train_loop
from nlp_project.vectorizers import fit_tfidf, transform_tfidf

set_seed()
FIG_DIR = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Load + preprocess (drop stopwords; TF-IDF benefits from a smaller, more discriminative vocab).
train_docs, train_labels, test_docs, test_labels, label_names = load_20ng(remove=True)
train_tokens = preprocess(train_docs, drop_stopwords=True)
test_tokens = preprocess(test_docs, drop_stopwords=True)

# TfidfVectorizer wants strings; rejoin our cleaned tokens.
train_strings = [" ".join(t) for t in train_tokens]
test_strings = [" ".join(t) for t in test_tokens]


In [ ]:
# Fit TF-IDF on train, transform test.
vec, X_train_full = fit_tfidf(train_strings, max_features=20_000, min_df=2)
X_test = transform_tfidf(vec, test_strings)
print(f"X_train_full: {X_train_full.shape}, X_test: {X_test.shape}")


In [ ]:
# Train/val split + dense conversion for the MLP.
# 11k x 20k floats fits comfortably in memory (~1.7 GB float32).
X_train_full_dense = X_train_full.toarray().astype(np.float32)
X_test_dense = X_test.toarray().astype(np.float32)

X_train, y_train, X_val, y_val = train_val_split(
    list(X_train_full_dense), train_labels, val_frac=0.1, seed=SEED,
)
X_train, X_val = np.asarray(X_train), np.asarray(X_val)

def make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y).long())
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, 64, shuffle=True)
val_loader = make_loader(X_val, y_val, 64, shuffle=False)
test_loader = make_loader(X_test_dense, test_labels, 64, shuffle=False)


In [ ]:
# Train and evaluate.
run = wandb.init(
    project="hslu-nalapro-q1",
    name="q1c-tfidf",
    config={"vectorizer": "tfidf", "max_features": 20_000,
            "hidden_dim": 256, "dropout": 0.3, "lr": 1e-3, "batch_size": 64},
)

model = MLP(in_dim=X_train.shape[1], hidden_dim=256, num_classes=20, dropout=0.3)
history = train_loop(
    model, train_loader, val_loader,
    epochs=50, lr=1e-3, device=DEVICE, wandb_run=run, patience=5,
)

metrics = evaluate(model, test_loader, label_names, device=DEVICE)
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"test macro-F1: {metrics['macro_f1']:.4f}")
plot_confusion(
    metrics["confusion_matrix"], label_names,
    save_path=FIG_DIR / "confusion_matrix_q1c.png",
    title="Q1c — TF-IDF",
)
run.log({"test_accuracy": metrics["accuracy"], "test_macro_f1": metrics["macro_f1"]})
run.finish()


## Notes for the report

- TF-IDF expects to beat Q1b's mean-pool baseline. Discuss whether it does, and why. Bag-of-words methods carry no semantic similarity, but they preserve all word identities (vs the lossy averaging in mean-pool).
- Confusion matrix shows whether residual errors are between semantically related classes (e.g. `talk.religion.misc` vs `soc.religion.christian`).
